# Video Question Answering with Temporal Grounding

Bilingual (Arabic / English) multimodal video QA system that answers natural-language questions about a video and grounds its answer in specific timestamps, with four explainability visualizations.

**How to run on Google Colab (recommended):**
1. **Runtime → Change runtime type → T4 GPU**
2. Add your Groq API key: **🔑 Secrets → Add → Name: `GROQ_API_KEY`** and paste your key from <https://console.groq.com/keys>. Toggle **Notebook access** on.
3. **Runtime → Run all.** First run downloads ~3 GB of weights (a few minutes).
4. The last cell launches a Gradio app and prints a public URL valid for 72 hours.

**GPU requirement:** the pipeline assumes a CUDA-capable GPU. On Colab the free T4 is sufficient.

## 1. Setup — install dependencies

This cell installs all Python packages and `ffmpeg`. It only needs to run once per Colab session.

In [ ]:
!pip install -q faster-whisper transformers sentence-transformers \
    FlagEmbedding grad-cam gradio yt-dlp moviepy \
    opencv-python python-dotenv groq matplotlib seaborn
!apt-get install -y ffmpeg > /dev/null

## 2. Imports and GPU check

In [ ]:
from __future__ import annotations

import os
import re
import math
import hashlib
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns  # noqa: F401  (imported for default styling)
from tqdm.auto import tqdm

# ML stack
from transformers import CLIPModel, CLIPProcessor
from sentence_transformers import SentenceTransformer
from faster_whisper import WhisperModel

# Grad-CAM
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Video / audio I/O
import yt_dlp

# LLM
from groq import Groq

# UI
import gradio as gr

DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE: torch.dtype = torch.float16 if DEVICE == "cuda" else torch.float32

if DEVICE == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"✅ GPU detected: {gpu_name} ({vram_gb:.1f} GB VRAM)")
else:
    print("⚠️  No GPU detected. The pipeline will run very slowly on CPU.")

print(f"   torch: {torch.__version__}")
print(f"   device: {DEVICE}, dtype: {DTYPE}")

## 3. Configuration

The Groq API key is read from Colab Secrets. To set it:

1. Click the **🔑 (key)** icon in the left sidebar.
2. Click **+ Add new secret**.
3. Name: `GROQ_API_KEY`. Value: paste your key from <https://console.groq.com/keys>.
4. Toggle **Notebook access** on.

If running locally, place the key in a `.env` file (see `.env.example`) — `python-dotenv` is loaded if present.

In [ ]:
@dataclass
class Config:
    """Hyperparameters for the video QA pipeline."""

    # Sampling
    FPS_SAMPLE: int = 1
    TRANSCRIPT_CHUNK_SECONDS: int = 10

    # Retrieval
    TOP_K_MOMENTS: int = 5
    VISUAL_WEIGHT: float = 0.5
    AUDIO_WEIGHT: float = 0.5
    RETRIEVAL_WINDOW_S: float = 1.0

    # Models
    WHISPER_MODEL: str = "large-v3"
    CLIP_MODEL: str = "openai/clip-vit-base-patch32"
    MULTILINGUAL_CLIP: str = "sentence-transformers/clip-ViT-B-32-multilingual-v1"
    TEXT_RETRIEVAL_MODEL: str = "BAAI/bge-m3"
    GROQ_MODEL: str = "llama-3.3-70b-versatile"

    # I/O
    WORK_DIR: str = "/content"


CONFIG = Config()


def load_groq_client() -> Groq:
    """Return a Groq client, reading the key from Colab Secrets, env, or .env.

    Returns:
        A configured `Groq` client.

    Raises:
        RuntimeError: If no API key can be found.
    """
    key: str | None = None
    # 1) Colab secrets
    try:
        from google.colab import userdata  # type: ignore
        key = userdata.get("GROQ_API_KEY")
    except Exception:
        pass
    # 2) .env (local runs)
    if not key:
        try:
            from dotenv import load_dotenv
            load_dotenv()
        except Exception:
            pass
        key = os.getenv("GROQ_API_KEY")
    if not key:
        raise RuntimeError(
            "GROQ_API_KEY is not set. On Colab, add it via the 🔑 Secrets panel. "
            "Locally, put it in a .env file or export it in your shell."
        )
    return Groq(api_key=key)


GROQ_CLIENT = load_groq_client()
print("✅ Groq client ready.")

## 4. Load models

Each model is loaded once and cached in a module-level dict. First-run downloads total ~3 GB and take 2-3 minutes on Colab; subsequent runs reuse the Hugging Face cache.

In [ ]:
_MODEL_CACHE: dict[str, Any] = {}


def get_clip() -> tuple[CLIPModel, CLIPProcessor]:
    """Load CLIP-ViT-B/32 and its processor (cached)."""
    if "clip" not in _MODEL_CACHE:
        model = CLIPModel.from_pretrained(CONFIG.CLIP_MODEL).to(DEVICE).eval()
        processor = CLIPProcessor.from_pretrained(CONFIG.CLIP_MODEL)
        _MODEL_CACHE["clip"] = (model, processor)
        print(f"✅ Loaded CLIP: {CONFIG.CLIP_MODEL}")
    return _MODEL_CACHE["clip"]


def get_multilingual_clip() -> SentenceTransformer:
    """Load the multilingual text encoder aligned with the CLIP image space."""
    if "mclip" not in _MODEL_CACHE:
        model = SentenceTransformer(CONFIG.MULTILINGUAL_CLIP, device=DEVICE)
        _MODEL_CACHE["mclip"] = model
        print(f"✅ Loaded multilingual CLIP: {CONFIG.MULTILINGUAL_CLIP}")
    return _MODEL_CACHE["mclip"]


def get_text_retriever() -> SentenceTransformer:
    """Load bge-m3 for multilingual transcript retrieval (cached)."""
    if "bge" not in _MODEL_CACHE:
        model = SentenceTransformer(CONFIG.TEXT_RETRIEVAL_MODEL, device=DEVICE)
        _MODEL_CACHE["bge"] = model
        print(f"✅ Loaded text retriever: {CONFIG.TEXT_RETRIEVAL_MODEL}")
    return _MODEL_CACHE["bge"]


def get_whisper() -> WhisperModel:
    """Load faster-whisper large-v3 in fp16 on the GPU (cached)."""
    if "whisper" not in _MODEL_CACHE:
        compute_type = "float16" if DEVICE == "cuda" else "int8"
        model = WhisperModel(CONFIG.WHISPER_MODEL, device=DEVICE, compute_type=compute_type)
        _MODEL_CACHE["whisper"] = model
        print(f"✅ Loaded Whisper: {CONFIG.WHISPER_MODEL} ({compute_type})")
    return _MODEL_CACHE["whisper"]


# Eager-load so the first user query is fast.
_ = get_clip()
_ = get_multilingual_clip()
_ = get_text_retriever()
_ = get_whisper()
print("✅ All models loaded.")

## 5. Video ingestion (file or URL)

`download_video` accepts either a local path or an HTTP(S) URL (YouTube, direct mp4, etc) and returns a local file path.

In [ ]:
class VideoDownloadError(RuntimeError):
    """Raised when a remote video URL cannot be downloaded.

    Distinct from generic errors so the UI can prompt the user to upload
    the file directly instead of pasting a URL.
    """


def is_url(s: str) -> bool:
    """Return True if `s` looks like an HTTP(S) URL."""
    return bool(re.match(r"^https?://", (s or "").strip()))


def download_video(source: str) -> str:
    """Resolve a video source to a local file path.

    Args:
        source: Either a local file path or an HTTP(S) URL (YouTube, direct mp4, etc).

    Returns:
        Absolute path to a local video file.

    Raises:
        ValueError: If `source` is empty.
        FileNotFoundError: If a local path does not exist.
        VideoDownloadError: If yt-dlp cannot fetch the URL (login required,
            geo block, IP block, removed video, etc.).
    """
    if not source:
        raise ValueError("Empty video source.")

    if not is_url(source):
        path = Path(source).expanduser().resolve()
        if not path.exists():
            raise FileNotFoundError(f"Video file not found: {path}")
        return str(path)

    Path(CONFIG.WORK_DIR).mkdir(parents=True, exist_ok=True)
    out_template = str(Path(CONFIG.WORK_DIR) / "downloaded_video.%(ext)s")
    ydl_opts = {
        "outtmpl": out_template,
        "format": "mp4/bestvideo[ext=mp4]+bestaudio[ext=m4a]/best",
        "quiet": True,
        "no_warnings": True,
        "noplaylist": True,
        "merge_output_format": "mp4",
        # Try the Android & iOS YouTube clients before the web client — this
        # often bypasses the "This video is not available" error returned to
        # Colab / data-center IPs by the web client.
        "extractor_args": {
            "youtube": {"player_client": ["android", "ios", "web"]},
        },
    }
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(source, download=True)
            filename = ydl.prepare_filename(info)
    except Exception as e:
        msg = str(e)
        host = re.sub(r"^https?://(www\.)?", "", source).split("/")[0]
        hint = (
            f"Could not download from {host}. This usually means the URL is "
            f"login-gated (Instagram, private YouTube, etc.) or the host is "
            f"blocking Colab's IP range. **Upload the video file directly via "
            f"the Inputs tab** — that always works.\n\nUnderlying error: {msg}"
        )
        raise VideoDownloadError(hint) from e

    return str(Path(filename).resolve())

## 6. Frame extraction (1 fps default)

Sample frames at a fixed rate using OpenCV. 1 fps is enough for typical retrieval queries and keeps CLIP inference fast.

In [ ]:
def extract_frames(video_path: str, fps: int = 1) -> list[tuple[float, np.ndarray]]:
    """Sample frames from a video at a fixed rate.

    Args:
        video_path: Path to the video.
        fps: Target sampling rate in frames per second.

    Returns:
        List of `(timestamp_seconds, rgb_frame)` tuples sorted by timestamp.

    Raises:
        RuntimeError: If the video cannot be opened.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    src_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    step = max(int(round(src_fps / max(fps, 1))), 1)

    frames: list[tuple[float, np.ndarray]] = []
    idx = 0
    pbar = tqdm(total=total, desc="Extracting frames", leave=False)
    while True:
        ret, frame_bgr = cap.read()
        if not ret:
            break
        if idx % step == 0:
            ts = idx / src_fps
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            frames.append((ts, frame_rgb))
        idx += 1
        pbar.update(1)
    pbar.close()
    cap.release()
    return frames


def get_video_duration(video_path: str) -> float:
    """Return the duration of `video_path` in seconds."""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    cap.release()
    return n / fps if fps > 0 else 0.0

## 7. Audio transcription with timestamps

`faster-whisper` (`large-v3`) auto-detects the language per chunk and returns word-level timestamps that are later used to align transcript chunks with video timestamps.

In [ ]:
def transcribe_audio(video_path: str) -> list[dict]:
    """Transcribe the audio track of a video with word-level timestamps.

    Args:
        video_path: Path to the video file.

    Returns:
        List of segments: `[{"start": float, "end": float, "text": str, "language": str}]`.
        Returns an empty list if Whisper fails (e.g., silent video).
    """
    model = get_whisper()
    try:
        segments_iter, info = model.transcribe(
            video_path,
            beam_size=5,
            vad_filter=True,
            word_timestamps=True,
        )
    except Exception as e:
        print(f"⚠️  Whisper failed: {e}. Returning empty transcript.")
        return []

    detected_lang = getattr(info, "language", "unknown") if info else "unknown"
    segments: list[dict] = []
    for seg in tqdm(segments_iter, desc="Transcribing", leave=False):
        text = (seg.text or "").strip()
        if not text:
            continue
        segments.append({
            "start": float(seg.start),
            "end": float(seg.end),
            "text": text,
            "language": detected_lang,
        })
    return segments

## 8. Multimodal indexing — embed frames and transcript chunks

Three small helpers: batch CLIP embeddings of frames, sliding-window grouping of transcript segments into ~10 s chunks, and bge-m3 embeddings of those chunks. All embeddings are L2-normalized so cosine similarity is just a dot product.

In [ ]:
@torch.no_grad()
def embed_frames(frames: list[tuple[float, np.ndarray]], batch_size: int = 32) -> torch.Tensor:
    """Compute L2-normalized CLIP image embeddings for every sampled frame.

    Args:
        frames: List of `(timestamp, rgb_array)` tuples from `extract_frames`.
        batch_size: Mini-batch size for CLIP inference.

    Returns:
        Tensor of shape `(N, D)` on `DEVICE`. Empty tensor if `frames` is empty.
    """
    if not frames:
        return torch.empty(0, 512, device=DEVICE)
    model, processor = get_clip()
    embeds: list[torch.Tensor] = []
    for i in tqdm(range(0, len(frames), batch_size), desc="Embedding frames", leave=False):
        batch = [Image.fromarray(f) for _, f in frames[i:i + batch_size]]
        inputs = processor(images=batch, return_tensors="pt").to(DEVICE)
        # Call the vision tower directly — `get_image_features` returns a non-tensor
        # (BaseModelOutputWithPooling) on some transformers builds.
        vision_outputs = model.vision_model(pixel_values=inputs["pixel_values"])
        feats = model.visual_projection(vision_outputs.pooler_output)
        feats = F.normalize(feats, dim=-1)
        embeds.append(feats)
    return torch.cat(embeds, dim=0)


def chunk_transcript(segments: list[dict], window_s: int) -> list[dict]:
    """Group transcript segments into ~`window_s`-second chunks.

    Args:
        segments: Output of `transcribe_audio`.
        window_s: Target window width in seconds.

    Returns:
        List of `{"start", "end", "text", "language"}` dicts.
    """
    if not segments:
        return []
    chunks: list[dict] = []
    cur_start = segments[0]["start"]
    cur_end = cur_start + window_s
    cur_texts: list[str] = []
    cur_lang = segments[0].get("language", "unknown")

    for seg in segments:
        if seg["start"] >= cur_end and cur_texts:
            chunks.append({
                "start": cur_start,
                "end": min(cur_end, seg["start"]),
                "text": " ".join(cur_texts).strip(),
                "language": cur_lang,
            })
            cur_start = seg["start"]
            cur_end = cur_start + window_s
            cur_texts = []
        cur_texts.append(seg["text"])

    if cur_texts:
        chunks.append({
            "start": cur_start,
            "end": cur_end,
            "text": " ".join(cur_texts).strip(),
            "language": cur_lang,
        })
    return chunks


@torch.no_grad()
def embed_text_chunks(chunks: list[dict]) -> torch.Tensor:
    """Embed transcript chunks with bge-m3 (L2-normalized).

    Args:
        chunks: Output of `chunk_transcript`.

    Returns:
        Tensor of shape `(N, 1024)` on `DEVICE`. Empty tensor if `chunks` is empty.
    """
    if not chunks:
        return torch.empty(0, 1024, device=DEVICE)
    model = get_text_retriever()
    texts = [c["text"] for c in chunks]
    embs = model.encode(
        texts,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return embs.to(DEVICE)

## 9. Multimodal retrieval

For every sampled timestamp the system computes a visual score (max similarity over nearby frames within `±RETRIEVAL_WINDOW_S`) and an audio score (max similarity over chunks containing the timestamp). The top-K combined moments are returned, deduplicated so they do not overlap.

In [ ]:
ARABIC_RE = re.compile(r"[\u0600-\u06FF]")


def detect_language(text: str) -> str:
    """Return `'ar'` if `text` contains any Arabic characters, else `'en'`."""
    return "ar" if ARABIC_RE.search(text or "") else "en"


@torch.no_grad()
def _embed_question_visual(question: str) -> torch.Tensor:
    """Embed a question into CLIP image space using multilingual-CLIP."""
    model = get_multilingual_clip()
    emb = model.encode([question], convert_to_tensor=True, normalize_embeddings=True)
    return emb.to(DEVICE)


@torch.no_grad()
def _embed_question_text(question: str) -> torch.Tensor:
    """Embed a question with bge-m3 for transcript matching."""
    model = get_text_retriever()
    emb = model.encode([question], convert_to_tensor=True, normalize_embeddings=True)
    return emb.to(DEVICE)


def _audio_score_for_timestamp(t: float, chunk_sims: torch.Tensor, chunks: list[dict]) -> float:
    """Return max similarity over chunks whose interval contains `t`."""
    if chunk_sims.numel() == 0:
        return 0.0
    best = 0.0
    for i, ch in enumerate(chunks):
        if ch["start"] <= t <= ch["end"]:
            best = max(best, float(chunk_sims[i].item()))
    return best


def _visual_score_for_timestamp(
    t: float,
    frame_sims: torch.Tensor,
    frames: list[tuple[float, np.ndarray]],
    window_s: float,
) -> tuple[float, int]:
    """Return `(max similarity, index of best frame)` within `±window_s` of `t`."""
    if frame_sims.numel() == 0:
        return 0.0, -1
    best = 0.0
    best_idx = -1
    for i, (ts, _) in enumerate(frames):
        if abs(ts - t) <= window_s:
            s = float(frame_sims[i].item())
            if s > best:
                best = s
                best_idx = i
    return best, best_idx


def retrieve_moments(
    question: str,
    frames: list[tuple[float, np.ndarray]],
    frame_embeds: torch.Tensor,
    chunks: list[dict],
    chunk_embeds: torch.Tensor,
    top_k: int = 5,
    visual_weight: float = 0.5,
    audio_weight: float = 0.5,
    window_s: float = 1.0,
) -> list[dict]:
    """Return the top-K most relevant moments for `question`.

    Args:
        question: Natural-language question (Arabic or English).
        frames: Output of `extract_frames`.
        frame_embeds: Output of `embed_frames`.
        chunks: Output of `chunk_transcript`.
        chunk_embeds: Output of `embed_text_chunks`.
        top_k: How many moments to return.
        visual_weight: Weight α on the visual score.
        audio_weight: Weight β on the audio score.
        window_s: Half-width of the ±window for visual matching, in seconds.

    Returns:
        List of dicts with keys `timestamp`, `visual_score`, `audio_score`,
        `combined`, `frame`, `transcript`, `frame_index`.
    """
    if not frames:
        return []

    q_visual = _embed_question_visual(question)
    q_text = _embed_question_text(question)

    if frame_embeds.numel():
        frame_sims = (frame_embeds @ q_visual.T).squeeze(-1).clamp(min=0.0)
    else:
        frame_sims = torch.zeros(0, device=DEVICE)

    if chunk_embeds.numel():
        chunk_sims = (chunk_embeds @ q_text.T).squeeze(-1).clamp(min=0.0)
    else:
        chunk_sims = torch.zeros(0, device=DEVICE)

    candidates: list[dict] = []
    for i, (ts, frame) in enumerate(frames):
        v_score, v_idx = _visual_score_for_timestamp(ts, frame_sims, frames, window_s)
        a_score = _audio_score_for_timestamp(ts, chunk_sims, chunks)
        if v_idx < 0:
            v_idx = i
        combined = visual_weight * v_score + audio_weight * a_score
        transcript = ""
        for ch in chunks:
            if ch["start"] <= ts <= ch["end"]:
                transcript = ch["text"]
                break
        candidates.append({
            "timestamp": float(ts),
            "visual_score": float(v_score),
            "audio_score": float(a_score),
            "combined": float(combined),
            "frame": frames[v_idx][1] if 0 <= v_idx < len(frames) else frame,
            "transcript": transcript,
            "frame_index": v_idx,
        })

    # Dedup by minimum spacing of `2 * window_s` so top-K do not overlap.
    candidates.sort(key=lambda c: c["combined"], reverse=True)
    selected: list[dict] = []
    for cand in candidates:
        if all(abs(cand["timestamp"] - s["timestamp"]) > window_s * 2 for s in selected):
            selected.append(cand)
        if len(selected) >= top_k:
            break
    return selected

## 10. Answer generation via Groq

The Groq LLM (`llama-3.3-70b-versatile`) is asked to answer using only the retrieved evidence and to reply in the same language as the question.

In [ ]:
def _format_seconds(t: float) -> str:
    """Format seconds as `mm:ss`."""
    m, s = divmod(int(round(t)), 60)
    return f"{m:02d}:{s:02d}"


def generate_answer(question: str, moments: list[dict], language: str) -> dict:
    """Generate a grounded answer with cited timestamps via Groq.

    Args:
        question: User's question.
        moments: Output of `retrieve_moments`.
        language: 'ar' or 'en'. The model is instructed to reply in this language.

    Returns:
        {"answer": str, "cited_timestamps": list[float]}.
    """
    if not moments:
        return {"answer": "No relevant moments were found in the video.",
                "cited_timestamps": []}

    evidence_lines: list[str] = []
    for m in moments:
        ts_str = _format_seconds(m["timestamp"])
        snippet = m["transcript"] or "(no speech in this window)"
        evidence_lines.append(
            f"- [{ts_str}] visual={m['visual_score']:.2f} audio={m['audio_score']:.2f} "
            f"transcript: {snippet}"
        )
    evidence = "\n".join(evidence_lines)

    reply_lang = "Arabic" if language == "ar" else "English"
    sys_msg = (
        "You answer questions about a video using ONLY the supplied evidence. "
        f"Reply in {reply_lang} with a short, precise answer and cite the timestamps "
        "your answer relies on in [mm:ss] format. If the evidence is insufficient, "
        "say so plainly."
    )

    user_msg = (
        f"Question: {question}\n\n"
        f"Evidence (top {len(moments)} moments, ranked):\n{evidence}\n\n"
        "Answer the question, citing timestamps."
    )

    try:
        resp = GROQ_CLIENT.chat.completions.create(
            model=CONFIG.GROQ_MODEL,
            messages=[
                {"role": "system", "content": sys_msg},
                {"role": "user", "content": user_msg},
            ],
            temperature=0.2,
            max_tokens=512,
        )
        answer = (resp.choices[0].message.content or "").strip()
    except Exception as e:
        answer = f"Groq API error: {e}\n\nRaw evidence:\n{evidence}"

    cited = [m["timestamp"] for m in moments]
    return {"answer": answer, "cited_timestamps": cited}


def translate_to_english(text: str) -> str:
    """Translate `text` to English via Groq (used to align non-English queries with CLIP)."""
    if detect_language(text) == "en":
        return text
    try:
        resp = GROQ_CLIENT.chat.completions.create(
            model=CONFIG.GROQ_MODEL,
            messages=[
                {"role": "system",
                 "content": "Translate the user text to English. Reply with the translation only."},
                {"role": "user", "content": text},
            ],
            temperature=0.0,
            max_tokens=128,
        )
        return (resp.choices[0].message.content or text).strip()
    except Exception:
        return text

## 11. XAI #1 — Timeline relevance plot

Plots per-second visual relevance (CLIP) and audio relevance (bge-m3) across the whole video. The top-K moments chosen for the answer are shaded green.

In [ ]:
@torch.no_grad()
def plot_timeline(
    question: str,
    frames: list[tuple[float, np.ndarray]],
    frame_embeds: torch.Tensor,
    chunks: list[dict],
    chunk_embeds: torch.Tensor,
    moments: list[dict],
    video_duration: float,
) -> plt.Figure:
    """Plot per-second visual and audio relevance over the whole video.

    Args:
        question: User question.
        frames, frame_embeds, chunks, chunk_embeds: As in `retrieve_moments`.
        moments: Top-K moments to highlight as shaded green bands.
        video_duration: Video length in seconds.

    Returns:
        Matplotlib figure.
    """
    q_visual = _embed_question_visual(question)
    q_text = _embed_question_text(question)

    frame_sims = (
        (frame_embeds @ q_visual.T).squeeze(-1).clamp(min=0.0).cpu().numpy()
        if frame_embeds.numel() else np.zeros(0)
    )
    chunk_sims = (
        (chunk_embeds @ q_text.T).squeeze(-1).clamp(min=0.0).cpu().numpy()
        if chunk_embeds.numel() else np.zeros(0)
    )

    frame_ts = np.array([t for t, _ in frames]) if frames else np.zeros(0)

    fig, ax = plt.subplots(figsize=(11, 4))
    if frame_sims.size:
        ax.plot(frame_ts, frame_sims, color="#1f77b4", linewidth=1.6,
                label="Visual relevance (CLIP)", alpha=0.9)
    for i, ch in enumerate(chunks):
        if i >= len(chunk_sims):
            break
        ax.fill_between([ch["start"], ch["end"]], 0, chunk_sims[i],
                        color="#ff7f0e", alpha=0.25,
                        label="Audio relevance (bge-m3)" if i == 0 else None)

    for m in moments:
        ax.axvspan(m["timestamp"] - 0.5, m["timestamp"] + 0.5,
                   color="green", alpha=0.18)

    y_max = max(1.0, float(np.max(frame_sims) if frame_sims.size else 0.0) * 1.1,
                float(np.max(chunk_sims) if chunk_sims.size else 0.0) * 1.1)
    ax.set_xlim(0, max(video_duration, 1.0))
    ax.set_ylim(0, y_max)
    ax.set_xlabel("Time (seconds)")
    ax.set_ylabel("Relevance")
    ax.set_title(f"Timeline relevance for: {question[:80]}")
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig

## 12. XAI #2 — Grad-CAM on the top frame

Saliency overlay showing which pixels in the top frame matched the question. Because CLIP's text encoder is English-only, Arabic questions are translated via Groq for this step; the UI shows both versions.

In [ ]:
class _CLIPSimilarityWrapper(nn.Module):
    """Wraps a CLIP model so it returns a single (B, 1) similarity score per image.

    This lets us reuse `pytorch-grad-cam`'s standard `ClassifierOutputTarget(0)`.
    """

    def __init__(self, clip_model: CLIPModel, text_features: torch.Tensor) -> None:
        super().__init__()
        self.clip_model = clip_model
        self.text_features = text_features  # (1, D), L2-normalized

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        vision_outputs = self.clip_model.vision_model(pixel_values=pixel_values)
        image_features = self.clip_model.visual_projection(vision_outputs.pooler_output)
        image_features = F.normalize(image_features, dim=-1)
        return image_features @ self.text_features.T  # (B, 1)


def gradcam_on_frame(frame: np.ndarray, question: str) -> tuple[np.ndarray, str]:
    """Compute a Grad-CAM heatmap overlay on `frame` for the question.

    Args:
        frame: RGB image, shape `(H, W, 3)`, dtype `uint8`.
        question: Original (possibly Arabic) question.

    Returns:
        Tuple `(heatmap_overlay_rgb, english_question_used)`.
    """
    model, processor = get_clip()
    en_question = translate_to_english(question)

    with torch.no_grad():
        text_inputs = processor(text=[en_question], return_tensors="pt",
                                padding=True, truncation=True).to(DEVICE)
        # Bypass get_text_features for the same compatibility reason as embed_frames.
        text_outputs = model.text_model(**text_inputs)
        text_features = model.text_projection(text_outputs.pooler_output)
        text_features = F.normalize(text_features, dim=-1)

    wrapper = _CLIPSimilarityWrapper(model, text_features).to(DEVICE).eval()

    pil = Image.fromarray(frame)
    pixel_values = processor(images=pil, return_tensors="pt")["pixel_values"].to(DEVICE)
    pixel_values.requires_grad_(True)

    target_layers = [model.vision_model.encoder.layers[-1].layer_norm1]

    def reshape_transform(tensor: torch.Tensor) -> torch.Tensor:
        # CLIP returns (B, tokens, dim) with a CLS token at position 0; drop it
        # and reshape the patch tokens into a square grid for Grad-CAM.
        result = tensor[:, 1:, :]
        n_tokens = result.shape[1]
        side = int(round(math.sqrt(n_tokens)))
        result = result.reshape(result.size(0), side, side, result.size(2))
        return result.permute(0, 3, 1, 2)

    try:
        cam = GradCAM(model=wrapper, target_layers=target_layers,
                      reshape_transform=reshape_transform)
        targets = [ClassifierOutputTarget(0)]
        grayscale = cam(input_tensor=pixel_values, targets=targets)[0]
    except Exception as e:
        print(f"⚠️  Grad-CAM failed: {e}. Returning the raw frame.")
        return frame, en_question

    img_resized = cv2.resize(frame, (grayscale.shape[1], grayscale.shape[0]))
    img_norm = img_resized.astype(np.float32) / 255.0
    overlay = show_cam_on_image(img_norm, grayscale, use_rgb=True)
    return overlay, en_question

## 13. XAI #3 — Frame-similarity grid

Grid of the top candidate frames, each annotated with its visual / audio / combined scores. The frame used in the answer gets a green border.

In [ ]:
def plot_frame_similarity_grid(
    top_moments: list[dict],
    question: str,
    chosen_index: int = 0,
) -> plt.Figure:
    """Show the top moments as a 2×3 grid of frames with their scores.

    Args:
        top_moments: Output of `retrieve_moments`.
        question: User question (used in the figure title).
        chosen_index: Index of the moment used in the answer (gets a green border).

    Returns:
        Matplotlib figure.
    """
    n = min(len(top_moments), 6)
    if n == 0:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.text(0.5, 0.5, "No moments to display.", ha="center", va="center")
        ax.axis("off")
        return fig

    rows = 2 if n > 3 else 1
    cols = 3 if n > 1 else 1
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3))
    axes = np.atleast_1d(axes).flatten()

    for i in range(rows * cols):
        ax = axes[i]
        if i < n:
            m = top_moments[i]
            ax.imshow(m["frame"])
            ax.set_title(
                f"#{i+1}  t={_format_seconds(m['timestamp'])}\n"
                f"V={m['visual_score']:.2f}  A={m['audio_score']:.2f}  "
                f"Σ={m['combined']:.2f}",
                fontsize=10,
            )
            if i == chosen_index:
                for spine in ax.spines.values():
                    spine.set_edgecolor("green")
                    spine.set_linewidth(4)
            ax.set_xticks([])
            ax.set_yticks([])
        else:
            ax.axis("off")

    fig.suptitle(f"Top candidate moments for: {question[:80]}", fontsize=12)
    plt.tight_layout()
    return fig

## 14. XAI #4 — Multimodal contribution chart

Stacked bar per top moment showing the **weighted** visual contribution (`α · visual_score`) vs the audio contribution (`β · audio_score`). This makes "did the answer come from sight or sound?" a one-glance question.

In [ ]:
def plot_modality_contribution(
    top_moments: list[dict],
    visual_weight: float,
    audio_weight: float,
) -> plt.Figure:
    """Stacked bar chart showing visual vs audio contribution to each top moment.

    Args:
        top_moments: Output of `retrieve_moments`.
        visual_weight: Fusion weight α.
        audio_weight: Fusion weight β.

    Returns:
        Matplotlib figure.
    """
    if not top_moments:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.text(0.5, 0.5, "No moments to display.", ha="center", va="center")
        ax.axis("off")
        return fig

    labels = [_format_seconds(m["timestamp"]) for m in top_moments]
    visual_vals = np.array([visual_weight * m["visual_score"] for m in top_moments])
    audio_vals = np.array([audio_weight * m["audio_score"] for m in top_moments])

    x = np.arange(len(labels))
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.bar(x, visual_vals, label=f"Visual (α={visual_weight})", color="#1f77b4")
    ax.bar(x, audio_vals, bottom=visual_vals,
           label=f"Audio (β={audio_weight})", color="#ff7f0e")

    for i in range(len(top_moments)):
        total = visual_vals[i] + audio_vals[i]
        ax.text(i, total + 0.01, f"{total:.2f}", ha="center", fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_xlabel("Moment timestamp (mm:ss)")
    ax.set_ylabel("Weighted score contribution")
    ax.set_title("Did the answer come from sight or sound? (per top moment)")
    ax.legend()
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    return fig

## 15. End-to-end pipeline

Orchestrates download → frames → transcription → embeddings → retrieval → answer → 4 XAI plots. Embeddings are cached in memory keyed by a SHA-1 of the video file, so follow-up questions on the same video are answered without re-indexing.

In [ ]:
_VIDEO_CACHE: dict[str, dict] = {}


def _video_hash(path: str) -> str:
    """SHA-1 over the file's first 1 MB plus its size — cheap and stable enough."""
    h = hashlib.sha1()
    p = Path(path)
    with p.open("rb") as f:
        h.update(f.read(1024 * 1024))
    h.update(str(p.stat().st_size).encode())
    return h.hexdigest()


def _index_video(video_path: str, progress: Any = None) -> dict:
    """Run the heavy indexing for a video (frames + transcript + embeddings) with caching."""
    key = _video_hash(video_path)
    if key in _VIDEO_CACHE:
        return _VIDEO_CACHE[key]

    def _step(p: float, msg: str) -> None:
        if progress is not None:
            progress(p, desc=msg)

    _step(0.10, "Extracting frames")
    frames = extract_frames(video_path, fps=CONFIG.FPS_SAMPLE)

    _step(0.30, "Transcribing audio (Whisper-large-v3)")
    segments = transcribe_audio(video_path)

    _step(0.55, "Embedding frames (CLIP)")
    frame_embeds = embed_frames(frames)

    _step(0.75, "Embedding transcript (bge-m3)")
    chunks = chunk_transcript(segments, CONFIG.TRANSCRIPT_CHUNK_SECONDS)
    chunk_embeds = embed_text_chunks(chunks)

    duration = get_video_duration(video_path)
    cached = {
        "video_path": video_path,
        "frames": frames,
        "frame_embeds": frame_embeds,
        "segments": segments,
        "chunks": chunks,
        "chunk_embeds": chunk_embeds,
        "duration": duration,
    }
    _VIDEO_CACHE[key] = cached
    return cached


def process_video_and_answer(
    video_source: str,
    question: str,
    progress: Any = None,
) -> dict:
    """Run the full pipeline and return the answer plus XAI artifacts.

    Args:
        video_source: Local path or URL to a video.
        question: Natural-language question (Arabic or English).
        progress: Optional `gr.Progress` object for UI updates.

    Returns:
        Dict with keys: `answer_md`, `top_frame`, `fig_timeline`, `fig_grid`,
        `fig_modality`, `language`, `english_question`.
    """
    if progress is not None:
        progress(0.02, desc="Resolving video")
    video_path = download_video(video_source)

    indexed = _index_video(video_path, progress=progress)

    if progress is not None:
        progress(0.85, desc="Retrieving moments")
    moments = retrieve_moments(
        question=question,
        frames=indexed["frames"],
        frame_embeds=indexed["frame_embeds"],
        chunks=indexed["chunks"],
        chunk_embeds=indexed["chunk_embeds"],
        top_k=CONFIG.TOP_K_MOMENTS,
        visual_weight=CONFIG.VISUAL_WEIGHT,
        audio_weight=CONFIG.AUDIO_WEIGHT,
        window_s=CONFIG.RETRIEVAL_WINDOW_S,
    )

    language = detect_language(question)

    if progress is not None:
        progress(0.92, desc="Generating answer with Groq")
    answer = generate_answer(question, moments, language=language)

    if progress is not None:
        progress(0.95, desc="Building XAI plots")

    if moments:
        top_frame_overlay, en_question = gradcam_on_frame(moments[0]["frame"], question)
    else:
        top_frame_overlay = np.zeros((224, 224, 3), dtype=np.uint8)
        en_question = translate_to_english(question)

    fig_timeline = plot_timeline(
        question=question,
        frames=indexed["frames"],
        frame_embeds=indexed["frame_embeds"],
        chunks=indexed["chunks"],
        chunk_embeds=indexed["chunk_embeds"],
        moments=moments,
        video_duration=indexed["duration"],
    )
    fig_grid = plot_frame_similarity_grid(moments, question, chosen_index=0)
    fig_modality = plot_modality_contribution(
        moments, CONFIG.VISUAL_WEIGHT, CONFIG.AUDIO_WEIGHT,
    )

    cited = " · ".join(_format_seconds(t) for t in answer["cited_timestamps"])
    answer_md = (
        f"**Answer / الإجابة**\n\n{answer['answer']}\n\n"
        f"---\n*Cited moments / اللحظات المُستشهد بها:* {cited}\n"
        f"*Detected question language:* `{language}` · "
        f"*English form used for Grad-CAM:* `{en_question}`"
    )

    if progress is not None:
        progress(1.0, desc="Done")

    return {
        "answer_md": answer_md,
        "top_frame": top_frame_overlay,
        "fig_timeline": fig_timeline,
        "fig_grid": fig_grid,
        "fig_modality": fig_modality,
        "language": language,
        "english_question": en_question,
    }

## 16. Gradio UI

Launches a public Gradio demo. The link is valid for 72 hours and can be opened from any device.

In [ ]:
def _gradio_handler(
    video_file: str | None,
    video_url: str,
    question: str,
    progress: gr.Progress = gr.Progress(),
) -> tuple[str, np.ndarray, plt.Figure, plt.Figure, plt.Figure]:
    """Bridge function called by the Gradio UI."""
    source = (video_url or "").strip() or video_file
    if not source:
        raise gr.Error("Please upload a video or paste a URL.")
    if not (question or "").strip():
        raise gr.Error("Please enter a question.")
    try:
        out = process_video_and_answer(source, question.strip(), progress=progress)
    except VideoDownloadError as e:
        # Surface the actionable hint without a stack trace.
        raise gr.Error(str(e)) from None
    return (
        out["answer_md"],
        out["top_frame"],
        out["fig_timeline"],
        out["fig_grid"],
        out["fig_modality"],
    )


with gr.Blocks(title="Video QA + Temporal Grounding", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# 🎥 Video QA with Temporal Grounding\n"
        "**Bilingual (العربية / English) multi-modal video question answering with explainable AI.**\n\n"
        "Ask any question about a video. The system fuses visual frames (CLIP) and audio transcript "
        "(Whisper + bge-m3), retrieves the most relevant moments, and explains its reasoning with four plots.\n\n"
        "> **Tip:** Direct file upload is the most reliable input. URL downloads (YouTube, Instagram, etc.) "
        "often fail on Colab because hosts block data-center IPs or require login."
    )

    with gr.Tab("Inputs / المدخلات"):
        with gr.Row():
            video_file = gr.Video(
                label="Upload video (recommended) / ارفع فيديو (موصى به)",
                sources=["upload"],
            )
            video_url = gr.Textbox(
                label="…or paste a video URL / أو الصق رابط فيديو",
                placeholder="https://www.youtube.com/watch?v=...  (may fail on Colab)",
                lines=1,
            )
        question = gr.Textbox(
            label="Question / السؤال",
            placeholder=(
                "e.g. What does the speaker show on screen at the start?  /  "
                "ماذا يعرض المتحدث على الشاشة في البداية؟"
            ),
            lines=2,
        )
        run_btn = gr.Button("Analyze / حلِّل", variant="primary")

    with gr.Tab("Answer / الإجابة"):
        answer_box = gr.Markdown()
        top_frame = gr.Image(
            label="Top frame with Grad-CAM / أهم لقطة مع تفسير بصري",
            type="numpy",
        )

    with gr.Tab("Explainability / التفسير"):
        fig_timeline = gr.Plot(label="1) Timeline relevance / الخط الزمني")
        fig_grid = gr.Plot(label="2) Top candidate frames / أعلى اللقطات المرشّحة")
        fig_modality = gr.Plot(
            label="3) Modality contribution (visual vs audio) / مساهمة كل وسيط"
        )

    run_btn.click(
        _gradio_handler,
        inputs=[video_file, video_url, question],
        outputs=[answer_box, top_frame, fig_timeline, fig_grid, fig_modality],
    )

demo.queue().launch(share=True, debug=True)

## 17. Notes

- The `share=True` link is valid for **72 hours**. Re-run the previous cell to regenerate it.
- Embeddings for a given video are cached in memory (`_VIDEO_CACHE`); follow-up questions on the same video are answered without re-indexing.
- For longer videos (> 5 min): increase `TRANSCRIPT_CHUNK_SECONDS` to 20 and lower `FPS_SAMPLE` to 0.5 in the config cell, then re-run from Cell 7 onward.
- If Grad-CAM raises a `RuntimeError` about non-leaf tensors, restart the runtime — this can happen if some earlier cell loaded the CLIP model under `torch.inference_mode`.
- If Groq returns rate-limit errors, wait 60 s and retry; the free tier resets quickly.
- To run another video without restarting, just paste a new URL or upload a new file in the Inputs tab and click **Analyze** again.